## 1. Cài đặt và import các thư viện

In [2]:
%pip install -q datasets qdrant-client langchain-google-genai langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

save_path = "../data/raw"
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"{save_path}")

In [4]:
from dotenv import load_dotenv
load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
# print(f"HF_TOKEN: {HF_TOKEN}")

In [5]:
from huggingface_hub import login
login(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 2. Load data từ Huggingface

In [ ]:
import os
import pandas as pd
from datasets import load_dataset

def download_legal_subset(subset):
    file_path = os.path.join(save_path, f"legal_{subset}.parquet")

    # Kiểm tra nếu file đã tồn tại thì bỏ qua
    if os.path.exists(file_path):
        print(f"Bộ {subset} đã tồn tại, bỏ qua.")
        return

    print(f"Đang tải bộ {subset}... (Sử dụng split='data')")
    try:
        ds = load_dataset(
            "th1nhng0/vietnamese-legal-documents",
            subset,
            split='data'
        )

        # Chuyển sang DataFrame
        df = pd.DataFrame(ds)

        # Lưu vào Drive dưới dạng Parquet
        df.to_parquet(file_path, index=False)
        print(f"Đã lưu {subset} thành công! ({len(df)} dòng)")

    except Exception as e:
        print(f"Lỗi khi tải bộ {subset}: {e}")

In [7]:
for s in ["metadata", "relationships"]:
    download_legal_subset(s)

Bộ metadata đã tồn tại, bỏ qua.
Bộ relationships đã tồn tại, bỏ qua.


*Load* riêng file content vì nội dung nhiều và dung lượng rất lớn

In [8]:
import os
import shutil
from huggingface_hub import hf_hub_download

# Config
save_path = "../data/raw"
os.makedirs(save_path, exist_ok=True)
dest = os.path.join(save_path, "legal_content.parquet")

if os.path.exists(dest):
    print("File exists.")
else:
    try:
        path_in_cache = hf_hub_download(
            repo_id="th1nhng0/vietnamese-legal-documents",
            filename="data/content.parquet",
            repo_type="dataset"
        )
        shutil.copy(path_in_cache, dest)
        print(f"Success: {dest}")
    except Exception as e:
        print(f"Error: {e}")

File exists.


In [9]:
import pandas as pd
import pyarrow.parquet as pq

file_path = "../data/raw/legal_content.parquet"
# file_path = "../data/raw/legal_metadata.parquet"
# file_path = "../data/raw/legal_relationships.parquet"

metadata = pq.read_metadata(file_path)
print(f"Số dòng thực tế: {metadata.num_rows}")

Số dòng thực tế: 178665


## 3. Gộp bảng content với bảng metadata


In [10]:
import pandas as pd

df_content = pd.read_parquet("../data/raw/legal_content.parquet")
df_meta = pd.read_parquet("../data/raw/legal_metadata.parquet")

df_content['id'] = df_content['id'].astype(str)
df_meta['id'] = df_meta['id'].astype(str)

df_final = pd.merge(df_content, df_meta, on='id', how='inner')

print(f"Số dòng sau khi gộp: {len(df_final)}")

Số dòng sau khi gộp: 176193


In [11]:
%pip install -q qdrant-client pandas pyarrow requests tqdm

Note: you may need to restart the kernel to use updated packages.


## 4. Xử lý phần content

### 4a. Xử lý và lọc theo date

In [12]:
import pandas as pd
from bs4 import BeautifulSoup
import datetime

# 1. Chuyển đổi ngày tháng và Sắp xếp
def parse_date(date_str):
    try:
        return pd.to_datetime(date_str, format='%d/%m/%Y')
    except:
        return pd.Timestamp.min

df_final['date_dt'] = df_final['ngay_ban_hanh'].apply(parse_date)
df_sorted = df_final.sort_values(by='date_dt', ascending=False).head(22000).copy()

In [13]:
df_sorted.info()

<class 'pandas.DataFrame'>
Index: 22000 entries, 158766 to 152906
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   22000 non-null  str   
 1   content_html         22000 non-null  str   
 2   title                22000 non-null  str   
 3   so_ky_hieu           22000 non-null  str   
 4   ngay_ban_hanh        22000 non-null  str   
 5   loai_van_ban         22000 non-null  str   
 6   ngay_co_hieu_luc     22000 non-null  str   
 7   ngay_het_hieu_luc    2542 non-null   str   
 8   nguon_thu_thap       8535 non-null   str   
 9   ngay_dang_cong_bao   22000 non-null  str   
 10  nganh                10807 non-null  str   
 11  linh_vuc             4909 non-null   str   
 12  co_quan_ban_hanh     22000 non-null  str   
 13  chuc_danh            21993 non-null  str   
 14  nguoi_ky             21878 non-null  str   
 15  pham_vi              20920 non-null  str   
 16  thong_tin_ap_d

In [14]:
print(df_sorted['content_html'].iloc[4])

<table class="detailcontent" width="100%" border="0" id="content">
                <tr>
                    <td colspan="3">
                        <div align="justify">
                            <p align="center">
	<strong>QUY</strong><strong>ẾT ĐỊNH</strong></p>
<p align="center">
	<strong>Phân cấp cho các cơ quan, đơn vị cấp tỉnh lựa chọn đơn vị sự nghiệp công</strong></p>
<p align="center">
	<strong>để cung ứng dịch vụ công theo hình thức giao nhiệm vụ, đặt hàng</strong></p>
<p align="center">
	<strong>hoặc đấu thầu (theo quy định pháp luật về đấu thầu)</strong></p>
<p align="center">
	<strong>trên địa bàn tỉnh Đồng Tháp</strong></p>
<p>
	 </p>
<p>
	<em>Căn cứ Luật Tổ chức chính quyền địa phương số 72/2025/QH15;</em></p>
<p>
	<em>Căn cứ Luật Ban hành văn bản quy phạm pháp luật số 64/2025/QH15 được sửa đổi, bổ sung bởi Luật số 87/2025/QH15;</em></p>
<p>
	<em>Căn cứ Luật Ngân sách nhà nước số 89/2025/QH15;</em></p>
<p>
	<em>Căn cứ Nghị định số 78/2025/NĐ-CP của Chính phủ ban hành 

### 4b. Xử lý phần content là dạng html

In [15]:
from bs4 import BeautifulSoup
import pandas as pd

from bs4 import BeautifulSoup

def clean_html_v3(text):
    if not text or pd.isna(text):
        return ""
    # Trích xuất text sạch
    soup = BeautifulSoup(text, "lxml")
    raw_text = soup.get_text(separator=" ")
    # Xử lý toàn bộ khoảng trắng, xuống dòng thừa
    clean_text = " ".join(raw_text.split())
    return clean_text

# 1. Làm sạch lại nội dung
df_sorted['content_clean'] = df_sorted['content_html'].apply(clean_html_v3)

# Xử lý các giá trị NaN còn lại (nếu có)
df_sorted = df_sorted.fillna("")

### 4c. Xử lý các dữ liệu còn thiếu khác

In [16]:
df_sorted['ngay_het_hieu_luc'] = df_sorted['ngay_het_hieu_luc'].fillna('31/12/2099')
# df_sorted.drop(columns=['thong_tin_ap_dung'], inplace=True)

fill_values = {
    'nganh': 'undetermined',
    'linh_vuc': 'undetermined',
    'nguon_thu_thap': 'Van ban phap luat',
    'chuc_danh': 'N/A',
    'nguoi_ky': 'N/A',
    'pham_vi': 'toan quoc'
}

df_sorted.fillna(value=fill_values, inplace=True)
print(df_sorted.isnull().sum())

id                     0
content_html           0
title                  0
so_ky_hieu             0
ngay_ban_hanh          0
loai_van_ban           0
ngay_co_hieu_luc       0
ngay_het_hieu_luc      0
nguon_thu_thap         0
ngay_dang_cong_bao     0
nganh                  0
linh_vuc               0
co_quan_ban_hanh       0
chuc_danh              0
nguoi_ky               0
pham_vi                0
thong_tin_ap_dung      0
tinh_trang_hieu_luc    0
date_dt                0
content_clean          0
dtype: int64


In [17]:
print(df_sorted.iloc[36])

id                                                                187554
content_html           <table class="detailcontent" width="100%" bord...
title                  Sửa đổi, bổ sung một số điều của Nghị định số ...
so_ky_hieu                                                 82/2026/NĐ-CP
ngay_ban_hanh                                                 20/03/2026
loai_van_ban                                                   Nghị định
ngay_co_hieu_luc                                              04/05/2026
ngay_het_hieu_luc                                                       
nguon_thu_thap                                                 Bản chính
ngay_dang_cong_bao                                                   ...
nganh                                                         Quốc phòng
linh_vuc                                        Xử lý vi phạm hành chính
co_quan_ban_hanh                                               Chính phủ
chuc_danh                                          

In [18]:
print(df_sorted['content_clean'].iloc[10])

QUYẾT ĐỊNH Phân cấp cho cơ quan, người có thẩm quyền thực hiện nhiệm vụ, quyền hạn được phân cấp trong lĩnh vực đất đai trên địa bàn tỉnh Lâm Đồng Căn cứ Luật Tổ chức chính quyền địa phương số 72/2025/QH15; Căn cứ Luật Ban hành văn bản quy phạm pháp luật số 64/2025/QH15; Luật Sửa đổi, bổ sung một số điều của Luật Ban hành văn bản quy phạm pháp luật số 87/2025/QH15; Căn cứ Luật Đất đai số 31/2024/QH15 được sửa đổi, bổ sung một số điều bởi các Luật số 43/2024/QH15, số 47/2024/QH15, số 58/2024/QH15, số 71/2025/QH15, số 84/2025/QH15, số 93/2025/QH15, số 95/2025/QH15, số 146/2025/QH15 và số 147/2025/QH15 (sau đây gọi là Luật Đất đai); Căn cứ Nghị quyết số 190/2025/QH15 ngày 19 tháng 02 năm 2025 của Quốc hội Quy định về xử lý một số vấn đề liên quan đến sắp xếp tổ chức bộ máy nhà nước; Căn cứ Nghị quyết số 254/2025/QH15 ngày 11 tháng 12 năm 2025 của Quốc hội quy định một số cơ chế, chính sách tháo gỡ khó khăn, vướng mắc trong tổ chức thi hành Luật Đất đai; Căn cứ Nghị định số 49/2026/NĐ-CP c

In [19]:
df_sorted = df_sorted[df_sorted['content_clean'].str.strip().astype(bool)]
print(f"Còn lại {len(df_sorted)} văn bản có nội dung.")

Còn lại 15507 văn bản có nội dung.


In [20]:
df_sorted = df_sorted.reset_index(drop=True)

In [21]:
def finalize_embedding_text(row):
    title = row['title'] if row['title'] else "Không có tiêu đề"
    so_hieu = row['so_ky_hieu'] if row['so_ky_hieu'] else "Không số hiệu"
    loai = row['loai_van_ban'] if row['loai_van_ban'] else ""
    content = row['content_clean']

    # Nếu nội dung quá ngắn (< 20 từ), ta coi như văn bản chỉ có trích yếu
    if len(content.split()) < 20:
        return f"Văn bản pháp luật: {title}. Số hiệu: {so_hieu}. Loại: {loai}. (Nội dung chi tiết đang cập nhật)"

    return f"Tiêu đề: {title}. Loại: {loai}. Số hiệu: {so_hieu}. Nội dung: {content}"

df_sorted['text_for_embedding'] = df_sorted.apply(finalize_embedding_text, axis=1)

In [22]:
print(df_sorted['text_for_embedding'].iloc[0])

Tiêu đề: Phân cấp cho các cơ quan, đơn vị cấp tỉnh lựa chọn đơn vị sự nghiệp công để cung ứng dịch vụ công theo hình thức giao nhiệm vụ, đặt hàng hoặc đấu thầu (theo quy định pháp luật về đấu thầu) trên địa bàn tỉnh Đồng Tháp. Loại: Quyết định. Số hiệu: 42/2026/QĐ-UBND. Nội dung: QUY ẾT ĐỊNH Phân cấp cho các cơ quan, đơn vị cấp tỉnh lựa chọn đơn vị sự nghiệp công để cung ứng dịch vụ công theo hình thức giao nhiệm vụ, đặt hàng hoặc đấu thầu (theo quy định pháp luật về đấu thầu) trên địa bàn tỉnh Đồng Tháp Căn cứ Luật Tổ chức chính quyền địa phương số 72/2025/QH15; Căn cứ Luật Ban hành văn bản quy phạm pháp luật số 64/2025/QH15 được sửa đổi, bổ sung bởi Luật số 87/2025/QH15; Căn cứ Luật Ngân sách nhà nước số 89/2025/QH15; Căn cứ Nghị định số 78/2025/NĐ-CP của Chính phủ ban hành quy định chi tiết một số điều và biện pháp để tổ chức, hướng dẫn thi hành Luật Ban hành văn bản quy phạm pháp luật được sửa đổi, bổ sung bởi Nghị định số 187/2025/NĐ-CP; Căn cứ Nghị định số 60/2021/NĐ-CP của Chính

In [23]:
df_sorted.to_parquet("../data/processed/legal_15k.parquet", index=False)